In [ ]:
"""
GPS Spoofing Detection — Binary XGBoost Training
Reframes as binary detection: legitimate (0) vs any attack (1)

Why binary instead of multi-class:
  - The deployment question is binary: trust this signal or not
  - Multi-class wastes capacity learning inter-attack boundaries
  - Binary directly optimizes the alarm decision
  - Nobody on the drone cares if it's simplistic vs sophisticated

Same 9 physics features as pass 2:
  DO, CP, EC, LC, PC, PIP, PQP, TCD, CN0
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    roc_auc_score,
    roc_curve,
)

import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

# ── 1. Configuration ──────────────────────────────────────────────────────────
DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("gps_spoofing_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.25

FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]

COLUMN_MAP = {"Output": "label"}


# ── 2. Load data ──────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    p = Path(path)
    if p.suffix == ".xlsx":
        df = pd.read_excel(p, engine="openpyxl")
    elif p.suffix == ".xls":
        df = pd.read_excel(p, engine="xlrd")
    else:
        df = pd.read_csv(p)

    rename = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
    if rename:
        df = df.rename(columns=rename)

    missing = [f for f in FEATURES + ["label"] if f not in df.columns]
    if missing:
        raise ValueError(
            f"Columns not found: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )

    print(f"Loaded {len(df):,} rows  |  {len(FEATURES)} physics features")
    return df


# ── 3. Binary label encoding ──────────────────────────────────────────────────
def make_binary_labels(df: pd.DataFrame) -> np.ndarray:
    """
    Collapses 4-class labels into binary:
      0 → 0  (legitimate)
      1 → 1  (simplistic  attack)
      2 → 1  (intermediate attack)
      3 → 1  (sophisticated attack)
    """
    y = (df["label"].astype(int) != 0).astype(int).values

    n_legit  = (y == 0).sum()
    n_attack = (y == 1).sum()
    print(f"\nBinary label distribution:")
    print(f"  Legitimate : {n_legit:>7,}  ({100*n_legit/len(y):.1f}%)")
    print(f"  Attack     : {n_attack:>7,}  ({100*n_attack/len(y):.1f}%)")
    print(f"  Imbalance ratio: {n_legit/n_attack:.2f}:1")
    return y


# ── 4. Split ──────────────────────────────────────────────────────────────────
def split_data(X, y):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, stratify=y_tv, random_state=RANDOM_STATE
    )
    print(f"\nSplit  →  train: {len(X_train):,}  |  "
          f"val: {len(X_val):,}  |  test: {len(X_test):,}")
    return X_train, X_val, X_test, y_train, y_val, y_test


# ── 5. Train ──────────────────────────────────────────────────────────────────
def train_model(X_train, y_train, X_val, y_val) -> xgb.XGBClassifier:
    """
    Binary objective: binary:logistic
    Outputs P(attack) ∈ [0,1] directly — no need to sum class probs.
    scale_pos_weight handles class imbalance (78% legit / 22% attack).
    """
    n_legit  = (y_train == 0).sum()
    n_attack = (y_train == 1).sum()
    scale_pw = n_legit / n_attack   # ~3.5, upweights attack class
    print(f"\nscale_pos_weight = {scale_pw:.2f}  "
          f"(compensates for {n_legit/n_attack:.1f}:1 imbalance)")

    model = xgb.XGBClassifier(
        n_estimators        = 1000,
        max_depth           = 6,
        learning_rate       = 0.05,
        subsample           = 0.8,
        colsample_bytree    = 0.8,
        min_child_weight    = 5,
        gamma               = 0.1,
        reg_alpha           = 0.1,
        reg_lambda          = 1.0,
        objective           = "binary:logistic",
        eval_metric         = "logloss",
        scale_pos_weight    = scale_pw,
        early_stopping_rounds = 30,
        tree_method         = "hist",
        random_state        = RANDOM_STATE,
        n_jobs              = -1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50,
    )
    print(f"\nBest iteration: {model.best_iteration}")
    return model


# ── 6. Evaluate ───────────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test):
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]   # P(attack)

    acc   = accuracy_score(y_test, y_pred)
    auroc = roc_auc_score(y_test, y_prob)

    print(f"\nTest accuracy : {acc:.4f}")
    print(f"AUROC         : {auroc:.4f}\n")
    print(classification_report(
        y_test, y_pred,
        target_names=["Legitimate", "Attack"]
    ))

    # ── Confusion matrix ──
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        cm, display_labels=["Legitimate", "Attack"]
    ).plot(ax=ax, cmap="Oranges", colorbar=False)
    ax.set_title("Binary GPS Spoofing Detection\nConfusion Matrix (9 physics features)")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "confusion_matrix_binary.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/confusion_matrix_binary.png")

    # ── ROC curve ──
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="#4C72B0", lw=2,
            label=f"XGBoost Binary  AUROC={auroc:.4f}")
    ax.plot([0,1], [0,1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate  (legitimate flagged as attack)")
    ax.set_ylabel("True Positive Rate  (attack detected)")
    ax.set_title("ROC Curve — Binary GPS Spoofing Detection")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "roc_curve_binary.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/roc_curve_binary.png")

    # ── Learning curve ──
    results = model.evals_result()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(results["validation_0"]["logloss"], label="Train", alpha=0.8)
    ax.plot(results["validation_1"]["logloss"], label="Val",   alpha=0.8)
    ax.axvline(model.best_iteration, color="red", ls="--",
               label=f"Best ({model.best_iteration})")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("logloss")
    ax.set_title("XGBoost Learning Curve (Binary)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "learning_curve_binary.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/learning_curve_binary.png")

    # ── FPR at key recall levels ──
    print("\nFPR at key recall (TPR) levels:")
    for target_tpr in [0.80, 0.90, 0.95, 0.99]:
        idx = np.where(tpr >= target_tpr)[0]
        if len(idx) > 0:
            print(f"  Recall={target_tpr:.0%}  →  "
                  f"FPR={fpr[idx[0]]:.4f}  "
                  f"(threshold={thresholds[idx[0]]:.4f})")

    return y_prob, auroc


# ── 7. Save artifacts ─────────────────────────────────────────────────────────
def save_artifacts(model, X_test, y_test, y_prob):
    model.save_model(OUTPUT_DIR / "xgb_binary.ubj")
    (OUTPUT_DIR / "feature_names.txt").write_text("\n".join(FEATURES))
    pd.DataFrame(X_test, columns=FEATURES).to_parquet(
        OUTPUT_DIR / "X_test_binary.parquet", index=False
    )
    np.save(OUTPUT_DIR / "y_test_binary.npy", y_test)
    np.save(OUTPUT_DIR / "y_prob_binary.npy", y_prob)

    print(f"\nArtifacts saved to ./{OUTPUT_DIR}/")
    print("  xgb_binary.ubj          — binary model")
    print("  feature_names.txt       — 9 physics features")
    print("  X_test_binary.parquet   — test features")
    print("  y_test_binary.npy       — true binary labels")
    print("  y_prob_binary.npy       — P(attack) scores for EU fusion")


# ── 8. Main ───────────────────────────────────────────────────────────────────
def main():
    df = load_data(DATA_PATH)

    X = df[FEATURES].values.astype(np.float32)
    y = make_binary_labels(df)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    print("\n── Training (Binary) ────────────────────────────────────────────")
    model = train_model(X_train, y_train, X_val, y_val)

    print("\n── Evaluation ───────────────────────────────────────────────────")
    y_prob, auroc = evaluate(model, X_test, y_test)

    save_artifacts(model, X_test, y_test, y_prob)

    print(f"""
── Summary ──────────────────────────────────────────────────────
Task    : Binary detection (legitimate vs any attack)
Features: 9 physics features (leaky features dropped)
AUROC   : {auroc:.4f}
Next    : run eu_fusion.py using xgb_binary.ubj + y_prob_binary.npy
""")


if __name__ == "__main__":
    main()

Loaded 510,530 rows  |  9 physics features

Binary label distribution:
  Legitimate : 397,825  (77.9%)
  Attack     : 112,705  (22.1%)
  Imbalance ratio: 3.53:1

Split  →  train: 306,318  |  val: 102,106  |  test: 102,106

── Training (Binary) ────────────────────────────────────────────

scale_pos_weight = 3.53  (compensates for 3.5:1 imbalance)
[0]	validation_0-logloss:0.67260	validation_1-logloss:0.67253
[50]	validation_0-logloss:0.38045	validation_1-logloss:0.37999
[100]	validation_0-logloss:0.31136	validation_1-logloss:0.31101
[150]	validation_0-logloss:0.25789	validation_1-logloss:0.25779
[200]	validation_0-logloss:0.22017	validation_1-logloss:0.22059
[250]	validation_0-logloss:0.19760	validation_1-logloss:0.19837
[300]	validation_0-logloss:0.18245	validation_1-logloss:0.18365
[350]	validation_0-logloss:0.16910	validation_1-logloss:0.17051
[400]	validation_0-logloss:0.15742	validation_1-logloss:0.15910
[450]	validation_0-logloss:0.14926	validation_1-logloss:0.15126
[500]	validati

In [ ]:
"""
GPS Spoofing Detection — Single XGBoost Baseline (No EU)
All 9 physics features, binary detection, no fusion layer.

Purpose: direct comparison against two-model EU system.
Saves to eu_architecture_outputs/ so test sets are identical.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
import xgboost as xgb
import matplotlib.pyplot as plt

DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("eu_architecture_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
FEATURES     = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]


def main():
    # ── Load ──────────────────────────────────────────────────────────────────
    df = pd.read_excel(DATA_PATH, engine="openpyxl")
    y  = (df["Output"].astype(int) != 0).astype(int).values
    X  = df[FEATURES].values.astype(np.float32)

    print(f"Loaded {len(df):,} rows")
    print(f"Legitimate: {(y==0).sum():,}  |  Attack: {(y==1).sum():,}")

    # ── Split — identical random_state to EU scripts ───────────────────────────
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=RANDOM_STATE
    )
    print(f"Split  →  train: {len(X_train):,}  |  "
          f"val: {len(X_val):,}  |  test: {len(X_test):,}")

    # ── Train ─────────────────────────────────────────────────────────────────
    scale_pw = (y_train==0).sum() / (y_train==1).sum()
    print(f"\nscale_pos_weight = {scale_pw:.2f}")

    model = xgb.XGBClassifier(
        n_estimators          = 2000,   # increased — previous run hit 999 limit
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        objective             = "binary:logistic",
        eval_metric           = "logloss",
        scale_pos_weight      = scale_pw,
        early_stopping_rounds = 30,
        tree_method           = "hist",
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100,
    )
    print(f"\nBest iteration: {model.best_iteration}")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auroc  = roc_auc_score(y_test, y_prob)

    print(f"\nAUROC : {auroc:.4f}")
    print(classification_report(
        y_test, y_pred,
        target_names=["Flight Allowed", "Breaker Tripped"]
    ))

    # Confusion matrix — same format as EU script
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(f"  True Negatives  (Safe flights continued)    : {cm[0][0]}")
    print(f"  False Positives (Nuisance Aborts)           : {cm[0][1]}")
    print(f"  False Negatives (Attacks that got through)  : {cm[1][0]}")
    print(f"  True Positives  (Attacks defeated)          : {cm[1][1]}")

    # FPR at key recall levels
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    print("\nFPR at key recall (TPR) levels:")
    for target_tpr in [0.75, 0.80, 0.90, 0.95, 0.99]:
        idx = np.where(tpr >= target_tpr)[0]
        if len(idx) > 0:
            print(f"  Recall={target_tpr:.0%}  →  "
                  f"FPR={fpr[idx[0]]:.4f}  "
                  f"threshold={thresholds[idx[0]]:.4f}")

    # ── ROC curve ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="#4C72B0", lw=2,
            label=f"Single XGBoost (9 features)  AUROC={auroc:.4f}")
    ax.plot([0,1], [0,1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate  (legitimate flagged as attack)")
    ax.set_ylabel("True Positive Rate  (attack detected)")
    ax.set_title("ROC Curve — Single XGBoost Baseline\n(All 9 physics features, no EU)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "roc_single_baseline.png", dpi=150)
    plt.close()
    print(f"\nSaved → {OUTPUT_DIR}/roc_single_baseline.png")

    # ── Save for comparison ───────────────────────────────────────────────────
    model.save_model(OUTPUT_DIR / "xgb_single_baseline.ubj")
    np.save(OUTPUT_DIR / "y_prob_single.npy", y_prob)
    np.save(OUTPUT_DIR / "y_test_shared.npy", y_test)   # shared ground truth
    print(f"Saved → {OUTPUT_DIR}/xgb_single_baseline.ubj")

    print(f"""
── Summary ──────────────────────────────────────────────────────
Model   : Single XGBoost, all 9 features, no EU
AUROC   : {auroc:.4f}
FN      : {cm[1][0]}  attacks missed
FP      : {cm[0][1]}  false alarms

Compare against EU system:
  Scenario A  recall=99%, FPR=17.7%  FN=274
  Scenario B  recall=75%, FPR=2.8%   FN=5585
────────────────────────────────────────────────────────────────
""")


if __name__ == "__main__":
    main()

Loaded 510,530 rows
Legitimate: 397,825  |  Attack: 112,705
Split  →  train: 306,318  |  val: 102,106  |  test: 102,106

scale_pos_weight = 3.53
[0]	validation_0-logloss:0.67260	validation_1-logloss:0.67253
[100]	validation_0-logloss:0.31136	validation_1-logloss:0.31101
[200]	validation_0-logloss:0.22017	validation_1-logloss:0.22059
[300]	validation_0-logloss:0.18245	validation_1-logloss:0.18365
[400]	validation_0-logloss:0.15742	validation_1-logloss:0.15910
[500]	validation_0-logloss:0.14229	validation_1-logloss:0.14459
[600]	validation_0-logloss:0.13150	validation_1-logloss:0.13440
[700]	validation_0-logloss:0.12411	validation_1-logloss:0.12774
[800]	validation_0-logloss:0.11868	validation_1-logloss:0.12330
[900]	validation_0-logloss:0.11391	validation_1-logloss:0.11952
[1000]	validation_0-logloss:0.11014	validation_1-logloss:0.11683
[1100]	validation_0-logloss:0.10738	validation_1-logloss:0.11530
[1200]	validation_0-logloss:0.10476	validation_1-logloss:0.11403
[1300]	validation_0-lo

In [ ]:
"""
GPS Spoofing Detection — System Comparison
Loads pre-computed probabilities from all models and compares them
on the identical test set at matched recall levels.

Requires (run these first):
  xgb_single_baseline.py     → y_prob_single.npy, y_test_shared.npy
  xgb_tracking_layer.py      → y_prob_tracking.npy
  xgb_observables_layer.py   → y_prob_observables.npy
"""

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
)

OUTPUT_DIR = Path("eu_architecture_outputs")

# ── EU parameters (must match eu_fusion script) ───────────────────────────────
SCENARIOS = [
    {
        "name":                "EU Scenario A — Balanced",
        "alpha":               0.60,
        "C_FN":                5.0,
        "C_FP":                5.0,
        "uncertainty_penalty": 1.0,
    },
    {
        "name":                "EU Scenario B — High Precision",
        "alpha":               0.50,
        "C_FN":                1.0,
        "C_FP":                3.0,
        "uncertainty_penalty": 0.5,
    },
]

TARGET_RECALLS = [0.75, 0.80, 0.90, 0.95, 0.99]


# ── Load artifacts ────────────────────────────────────────────────────────────
def load_artifacts():
    try:
        y_test         = np.load(OUTPUT_DIR / "y_test_shared.npy")
        p_single       = np.load(OUTPUT_DIR / "y_prob_single.npy")
        p_tracking     = np.load(OUTPUT_DIR / "y_prob_tracking.npy")
        p_observables  = np.load(OUTPUT_DIR / "y_prob_observables.npy")
    except FileNotFoundError as e:
        print(f"Missing artifact: {e}")
        print("Run xgb_single_baseline.py, xgb_tracking_layer.py, "
              "xgb_observables_layer.py first.")
        return None

    print(f"Test set: {len(y_test):,} samples  "
          f"|  Legitimate: {(y_test==0).sum():,}  "
          f"|  Attack: {(y_test==1).sum():,}")
    return y_test, p_single, p_tracking, p_observables


# ── EU fusion ─────────────────────────────────────────────────────────────────
def apply_eu(p_tracking, p_observables, alpha, C_FN, C_FP, uncertainty_penalty):
    """
    Returns fused probability and per-sample adaptive threshold.
    disagreement between layers dynamically lowers τ* for ambiguous samples.
    """
    p_fused      = alpha * p_tracking + (1 - alpha) * p_observables
    disagreement = np.abs(p_tracking - p_observables)
    dynamic_C_FN = C_FN * (1.0 + uncertainty_penalty * disagreement)
    tau          = C_FP / (C_FP + dynamic_C_FN)
    y_pred       = (p_fused >= tau).astype(int)
    return p_fused, tau, y_pred


# ── FPR at target recall ──────────────────────────────────────────────────────
def fpr_at_recall(y_true, y_prob, target_recall):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    idx = np.where(tpr >= target_recall)[0]
    if len(idx) == 0:
        return None, None
    return fpr[idx[0]], thresholds[idx[0]]


# ── Confusion matrix summary ──────────────────────────────────────────────────
def cm_summary(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    recall    = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr_val   = fp / (fp + tn)
    print(f"\n  {name}")
    print(f"    TP={tp:>6,}  FP={fp:>6,}  TN={tn:>6,}  FN={fn:>6,}")
    print(f"    Recall={recall:.4f}  Precision={precision:.4f}  FPR={fpr_val:.4f}")
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "recall": recall, "precision": precision, "fpr": fpr_val}


# ── Main comparison ───────────────────────────────────────────────────────────
def main():
    result = load_artifacts()
    if result is None:
        return
    y_test, p_single, p_tracking, p_observables = result

    print("\n" + "="*65)
    print(" AUROC COMPARISON")
    print("="*65)

    auroc_single      = roc_auc_score(y_test, p_single)
    auroc_tracking    = roc_auc_score(y_test, p_tracking)
    auroc_observables = roc_auc_score(y_test, p_observables)

    print(f"  Single XGBoost (9 features)     : {auroc_single:.4f}")
    print(f"  Tracking layer (6 features)     : {auroc_tracking:.4f}")
    print(f"  Observables layer (3 features)  : {auroc_observables:.4f}")

    # EU fused AUROC
    for sc in SCENARIOS:
        p_fused, tau, _ = apply_eu(
            p_tracking, p_observables,
            sc["alpha"], sc["C_FN"], sc["C_FP"], sc["uncertainty_penalty"]
        )
        auroc_eu = roc_auc_score(y_test, p_fused)
        print(f"  {sc['name']:<35}: {auroc_eu:.4f}")

    print("\n" + "="*65)
    print(" FPR AT MATCHED RECALL LEVELS")
    print("="*65)
    print(f"  {'System':<40}  " +
          "  ".join(f"R={r:.0%}" for r in TARGET_RECALLS))
    print("  " + "-"*62)

    systems = {
        "Single XGBoost (9 features)": p_single,
    }
    for sc in SCENARIOS:
        p_fused, _, _ = apply_eu(
            p_tracking, p_observables,
            sc["alpha"], sc["C_FN"], sc["C_FP"], sc["uncertainty_penalty"]
        )
        systems[sc["name"]] = p_fused

    for name, probs in systems.items():
        row = f"  {name:<40}"
        for r in TARGET_RECALLS:
            fpr_val, _ = fpr_at_recall(y_test, probs, r)
            row += f"  {fpr_val:.3f}  " if fpr_val is not None else "   N/A  "
        print(row)

    print("\n" + "="*65)
    print(" CONFUSION MATRICES AT DEFAULT THRESHOLD (0.5)")
    print("="*65)
    results = {}
    results["Single XGBoost"] = cm_summary(
        y_test, (p_single >= 0.5).astype(int), "Single XGBoost (threshold=0.5)"
    )
    for sc in SCENARIOS:
        p_fused, tau, y_pred = apply_eu(
            p_tracking, p_observables,
            sc["alpha"], sc["C_FN"], sc["C_FP"], sc["uncertainty_penalty"]
        )
        results[sc["name"]] = cm_summary(y_test, y_pred, sc["name"])

    print("\n" + "="*65)
    print(" WHAT THE TWO-MODEL EU SYSTEM GIVES OVER SINGLE MODEL")
    print("="*65)
    s = results["Single XGBoost"]
    for sc in SCENARIOS:
        e = results[sc["name"]]
        fn_diff  = s["fn"]  - e["fn"]
        fp_diff  = e["fp"]  - s["fp"]
        print(f"\n  vs {sc['name']}:")
        print(f"    FN change : {fn_diff:+,}  "
              f"({'fewer' if fn_diff > 0 else 'more'} missed attacks)")
        print(f"    FP change : {fp_diff:+,}  "
              f"({'more' if fp_diff > 0 else 'fewer'} false alarms)")
        print(f"    Recall    : {s['recall']:.4f} → {e['recall']:.4f}")
        print(f"    FPR       : {s['fpr']:.4f} → {e['fpr']:.4f}")

    # ── ROC overlay plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 7))

    fpr_s, tpr_s, _ = roc_curve(y_test, p_single)
    ax.plot(fpr_s, tpr_s, lw=2,
            label=f"Single XGBoost (9 features)  AUROC={auroc_single:.4f}")

    colors = ["#FF5722", "#9C27B0"]
    for sc, color in zip(SCENARIOS, colors):
        p_fused, _, _ = apply_eu(
            p_tracking, p_observables,
            sc["alpha"], sc["C_FN"], sc["C_FP"], sc["uncertainty_penalty"]
        )
        fpr_e, tpr_e, _ = roc_curve(y_test, p_fused)
        auroc_e = roc_auc_score(y_test, p_fused)
        ax.plot(fpr_e, tpr_e, lw=2, color=color,
                label=f"{sc['name']}  AUROC={auroc_e:.4f}")

    ax.plot([0,1], [0,1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Comparison — Single XGBoost vs Two-Model EU System")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 0.3)   # zoom in on relevant FPR range
    plt.tight_layout()
    path = OUTPUT_DIR / "roc_comparison.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"\nSaved → {path}")


if __name__ == "__main__":
    main()

Test set: 102,106 samples  |  Legitimate: 79,565  |  Attack: 22,541

 AUROC COMPARISON
  Single XGBoost (9 features)     : 0.9892
  Tracking layer (6 features)     : 0.8938
  Observables layer (3 features)  : 0.9897
  EU Scenario A — Balanced           : 0.9758
  EU Scenario B — High Precision     : 0.9817

 FPR AT MATCHED RECALL LEVELS
  System                                    R=75%  R=80%  R=90%  R=95%  R=99%
  --------------------------------------------------------------
  Single XGBoost (9 features)               0.002    0.024    0.052    0.061    0.068  
  EU Scenario A — Balanced                  0.040    0.049    0.068    0.100    0.175  
  EU Scenario B — High Precision            0.033    0.043    0.060    0.070    0.103  

 CONFUSION MATRICES AT DEFAULT THRESHOLD (0.5)

  Single XGBoost (threshold=0.5)
    TP=22,343  FP= 5,449  TN=74,116  FN=   198
    Recall=0.9912  Precision=0.8039  FPR=0.0685

  EU Scenario A — Balanced
    TP=22,267  FP=14,060  TN=65,505  FN=   274
  

In [ ]:
"""
GPS Spoofing Detection — Final Binary XGBoost
Single model on 9 physics features, binary detection.
Output: P(attack) fed into EU decision layer.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib

DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("final_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
FEATURES     = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]


def load_data():
    p  = Path(DATA_PATH)
    df = pd.read_excel(p, engine="openpyxl") if p.suffix == ".xlsx" else pd.read_csv(p)
    if "Output" in df.columns:
        df = df.rename(columns={"Output": "label"})

    X = df[FEATURES].values.astype(np.float32)
    y = (df["label"].astype(int) != 0).astype(int).values

    print(f"Loaded {len(df):,} rows")
    print(f"Legitimate : {(y==0).sum():,}  ({100*(y==0).mean():.1f}%)")
    print(f"Attack     : {(y==1).sum():,}  ({100*(y==1).mean():.1f}%)")
    return X, y


def split_data(X, y):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=RANDOM_STATE
    )
    print(f"\nSplit  →  train: {len(X_train):,}  |  "
          f"val: {len(X_val):,}  |  test: {len(X_test):,}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def train(X_train, y_train, X_val, y_val):
    scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
    print(f"\nscale_pos_weight = {scale_pw:.2f}")

    model = xgb.XGBClassifier(
        n_estimators          = 2000,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        objective             = "binary:logistic",
        eval_metric           = "logloss",
        scale_pos_weight      = scale_pw,
        early_stopping_rounds = 30,
        tree_method           = "hist",
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100,
    )
    print(f"Best iteration: {model.best_iteration}")
    return model


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auroc  = roc_auc_score(y_test, y_prob)

    print(f"\nAUROC : {auroc:.4f}")
    print(classification_report(
        y_test, y_pred,
        target_names=["Legitimate", "Attack"]
    ))

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}")

    # FPR at key recall levels
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    print("\nFPR at key recall levels (default threshold=0.5):")
    for r in [0.75, 0.80, 0.90, 0.95, 0.99]:
        idx = np.where(tpr >= r)[0]
        if len(idx):
            print(f"  Recall={r:.0%}  FPR={fpr[idx[0]]:.4f}  "
                  f"threshold={thresholds[idx[0]]:.4f}")

    # Learning curve
    results = model.evals_result()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(results["validation_0"]["logloss"], label="Train", alpha=0.8)
    ax.plot(results["validation_1"]["logloss"], label="Val",   alpha=0.8)
    ax.axvline(model.best_iteration, color="red", ls="--",
               label=f"Best iter ({model.best_iteration})")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("Log-loss")
    ax.set_title("XGBoost Learning Curve")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "learning_curve.png", dpi=150)
    plt.close()
    print(f"\nSaved → {OUTPUT_DIR}/learning_curve.png")

    return y_prob, fpr, tpr, thresholds, auroc


def save_artifacts(model, X_test, y_test, y_prob):
    model.save_model(OUTPUT_DIR / "xgb_final.ubj")
    pd.DataFrame(X_test, columns=FEATURES).to_parquet(
        OUTPUT_DIR / "X_test.parquet", index=False
    )
    np.save(OUTPUT_DIR / "y_test.npy",    y_test)
    np.save(OUTPUT_DIR / "y_prob.npy",    y_prob)
    (OUTPUT_DIR / "feature_names.txt").write_text("\n".join(FEATURES))

    print(f"\nArtifacts saved to ./{OUTPUT_DIR}/")
    print("  xgb_final.ubj      — trained model")
    print("  X_test.parquet     — test features")
    print("  y_test.npy         — true binary labels")
    print("  y_prob.npy         — P(attack) scores → feed into eu_decision.py")
    print("  feature_names.txt  — feature list")


def main():
    X, y = load_data()
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    print("\n── Training ─────────────────────────────────────────────────────")
    model = train(X_train, y_train, X_val, y_val)

    print("\n── Evaluation ───────────────────────────────────────────────────")
    y_prob, fpr, tpr, thresholds, auroc = evaluate(model, X_test, y_test)

    save_artifacts(model, X_test, y_test, y_prob)
    print(f"\nDone. AUROC={auroc:.4f}. Run eu_decision.py next.")


if __name__ == "__main__":
    main()

Loaded 510,530 rows
Legitimate : 397,825  (77.9%)
Attack     : 112,705  (22.1%)

Split  →  train: 306,318  |  val: 102,106  |  test: 102,106

── Training ─────────────────────────────────────────────────────

scale_pos_weight = 3.53
[0]	validation_0-logloss:0.67260	validation_1-logloss:0.67253
[100]	validation_0-logloss:0.31136	validation_1-logloss:0.31101
[200]	validation_0-logloss:0.22017	validation_1-logloss:0.22059
[300]	validation_0-logloss:0.18245	validation_1-logloss:0.18365
[400]	validation_0-logloss:0.15742	validation_1-logloss:0.15910
[500]	validation_0-logloss:0.14229	validation_1-logloss:0.14459
[600]	validation_0-logloss:0.13150	validation_1-logloss:0.13440
[700]	validation_0-logloss:0.12411	validation_1-logloss:0.12774
[800]	validation_0-logloss:0.11868	validation_1-logloss:0.12330
[900]	validation_0-logloss:0.11391	validation_1-logloss:0.11952
[1000]	validation_0-logloss:0.11014	validation_1-logloss:0.11683
[1100]	validation_0-logloss:0.10738	validation_1-logloss:0.11530

In [ ]:
"""
GPS Spoofing Detection — EU Decision Layer
Loads P(attack) from XGBoost and applies EU-derived adaptive threshold τ*.

EU framework:
  E[Alarm] = P(attack) × U(TP) + (1-P(attack)) × U(FP)
  E[Pass]  = P(attack) × U(FN) + (1-P(attack)) × U(TN)

  Alarm if E[Alarm] >= E[Pass]

  Solving analytically:
  τ* = (U(TN) - U(FP)) / (U(TP) - U(FN) + U(TN) - U(FP))

Utility value interpretation:
  U(TP) — reward for catching an attack
  U(TN) — reward for correctly passing a legitimate signal
  U(FP) — penalty for false alarm (legitimate flagged as attack)
  U(FN) — penalty for missed attack (most critical parameter)

Key insight: τ* shifts the operating point on the ROC curve
analytically — no retraining, no grid search, no validation set.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
)

OUTPUT_DIR = Path("final_outputs")

# ── Deployment scenarios ───────────────────────────────────────────────────────
# U(FN) is the critical parameter — cost of missing an attack.
# U(TP), U(TN), U(FP) held constant across scenarios for clean comparison.
# Only U(FN) changes to reflect deployment risk tolerance.

UTILITY_CONFIGS = [
    {
        "name":    "Military Drone",
        "desc":    "Missing attack is catastrophic — very low threshold",
        "color":   "#D32F2F",
        "U_TP":    10,
        "U_TN":    1,
        "U_FP":   -2,
        "U_FN":   -200,
    },
    {
        "name":    "Civilian UAV",
        "desc":    "Balanced — moderate penalty for missed attack",
        "color":   "#F57C00",
        "U_TP":    10,
        "U_TN":    1,
        "U_FP":   -2,
        "U_FN":   -50,
    },
    {
        "name":    "Survey Drone",
        "desc":    "Conservative — false alarms are costly too",
        "color":   "#388E3C",
        "U_TP":    10,
        "U_TN":    1,
        "U_FP":   -2,
        "U_FN":   -10,
    },
    {
        "name":    "Research / Low Risk",
        "desc":    "High precision — minimize nuisance alarms",
        "color":   "#1565C0",
        "U_TP":    10,
        "U_TN":    1,
        "U_FP":   -2,
        "U_FN":   -5,
    },
]


# ── Core EU functions ─────────────────────────────────────────────────────────
def compute_tau(U_TP, U_TN, U_FP, U_FN) -> float:
    """
    Analytically derives the optimal alarm threshold from utility values.
    τ* is the P(attack) crossover where E[Alarm] = E[Pass].
    """
    return (U_TN - U_FP) / (U_TP - U_FN + U_TN - U_FP)


def apply_threshold(y_prob, tau) -> np.ndarray:
    return (y_prob >= tau).astype(int)


def get_operating_point(fpr, tpr, thresholds, tau):
    """Finds the ROC operating point closest to τ*."""
    idx = np.argmin(np.abs(thresholds - tau))
    return fpr[idx], tpr[idx], thresholds[idx]


# ── Per-scenario evaluation ───────────────────────────────────────────────────
def evaluate_scenario(cfg, y_test, y_prob, fpr, tpr, thresholds):
    tau   = compute_tau(cfg["U_TP"], cfg["U_TN"], cfg["U_FP"], cfg["U_FN"])
    y_pred = apply_threshold(y_prob, tau)

    cm            = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    recall        = tp / (tp + fn)
    precision     = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr_val       = fp / (fp + tn)

    # Expected utility at this operating point
    p_attack = y_prob.mean()
    eu_alarm = p_attack * cfg["U_TP"] + (1-p_attack) * cfg["U_FP"]
    eu_pass  = p_attack * cfg["U_FN"] + (1-p_attack) * cfg["U_TN"]

    op_fpr, op_tpr, _ = get_operating_point(fpr, tpr, thresholds, tau)

    print(f"\n{'─'*55}")
    print(f" {cfg['name']}")
    print(f" {cfg['desc']}")
    print(f"{'─'*55}")
    print(f"  Utility values : U(TP)={cfg['U_TP']:>5}  U(TN)={cfg['U_TN']:>4}  "
          f"U(FP)={cfg['U_FP']:>4}  U(FN)={cfg['U_FN']:>5}")
    print(f"  τ* (threshold) : {tau:.4f}")
    print(f"  E[Alarm]       : {eu_alarm:.4f}")
    print(f"  E[Pass]        : {eu_pass:.4f}")
    print(f"  Decision       : alarm if P(attack) ≥ {tau:.4f}")
    print(f"\n  Results:")
    print(f"    TP (attacks caught)   : {tp:>6,}")
    print(f"    FN (attacks missed)   : {fn:>6,}  ← {fn/(tp+fn)*100:.1f}% miss rate")
    print(f"    FP (false alarms)     : {fp:>6,}  ← {fpr_val*100:.2f}% of legitimate")
    print(f"    TN (correct passes)   : {tn:>6,}")
    print(f"    Recall                : {recall:.4f}")
    print(f"    Precision             : {precision:.4f}")
    print(f"    FPR                   : {fpr_val:.4f}")

    return {
        "name": cfg["name"], "tau": tau,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "recall": recall, "precision": precision, "fpr": fpr_val,
        "op_fpr": op_fpr, "op_tpr": op_tpr,
        "color": cfg["color"],
    }


# ── Plots ─────────────────────────────────────────────────────────────────────
def plot_roc_with_operating_points(fpr, tpr, auroc, scenario_results):
    fig, ax = plt.subplots(figsize=(8, 7))

    # ROC curve
    ax.plot(fpr, tpr, color="#4C72B0", lw=2.5,
            label=f"Binary XGBoost  AUROC={auroc:.4f}", zorder=2)
    ax.plot([0,1], [0,1], "k--", lw=1, label="Random", zorder=1)

    # Operating points
    for r in scenario_results:
        ax.scatter(r["op_fpr"], r["op_tpr"],
                   color=r["color"], s=120, zorder=5,
                   label=f"{r['name']}  τ*={r['tau']:.3f}  "
                         f"R={r['recall']:.3f}  FPR={r['fpr']:.3f}")
        ax.annotate(
            r["name"],
            xy=(r["op_fpr"], r["op_tpr"]),
            xytext=(r["op_fpr"]+0.02, r["op_tpr"]-0.03),
            fontsize=8, color=r["color"],
        )

    ax.set_xlabel("False Positive Rate  (legitimate flagged as attack)")
    ax.set_ylabel("True Positive Rate  (attack detected)")
    ax.set_title("ROC Curve with EU-Derived Operating Points\n"
                 "Each point = analytically derived τ* for deployment context")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlim(-0.01, 0.35)
    ax.set_ylim(0.5, 1.01)
    plt.tight_layout()
    path = OUTPUT_DIR / "roc_eu_operating_points.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"\nSaved → {path}")


def plot_tau_vs_fn_cost(scenario_results):
    """Shows how τ* shifts as U(FN) changes — key paper figure."""
    fn_costs = [200, 100, 50, 20, 10, 5, 2]
    taus     = [compute_tau(10, 1, -2, -c) for c in fn_costs]
    recalls  = []
    fprs     = []

    y_test = np.load(OUTPUT_DIR / "y_test.npy")
    y_prob = np.load(OUTPUT_DIR / "y_prob.npy")
    fpr_c, tpr_c, thresh_c = roc_curve(y_test, y_prob)

    for tau in taus:
        y_pred = apply_threshold(y_prob, tau)
        cm     = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        recalls.append(tp / (tp + fn))
        fprs.append(fp / (fp + tn))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # τ* vs |U(FN)|
    ax = axes[0]
    ax.plot(fn_costs, taus, marker="o", color="#4C72B0")
    ax.set_xlabel("|U(FN)|  (cost of missed attack)")
    ax.set_ylabel("τ*  (alarm threshold)")
    ax.set_title("τ* vs False Negative Cost\n(higher FN cost → lower threshold)")
    ax.invert_xaxis()

    # Recall vs |U(FN)|
    ax = axes[1]
    ax.plot(fn_costs, recalls, marker="o", color="#DD8452")
    ax.set_xlabel("|U(FN)|  (cost of missed attack)")
    ax.set_ylabel("Recall  (attack detection rate)")
    ax.set_title("Recall vs False Negative Cost")
    ax.set_ylim(0, 1.05)
    ax.invert_xaxis()

    # FPR vs |U(FN)|
    ax = axes[2]
    ax.plot(fn_costs, fprs, marker="o", color="#2CA02C")
    ax.set_xlabel("|U(FN)|  (cost of missed attack)")
    ax.set_ylabel("FPR  (false alarm rate)")
    ax.set_title("FPR vs False Negative Cost\n(lower τ* → more false alarms)")
    ax.invert_xaxis()

    fig.suptitle(
        "EU Adaptive Threshold — Effect of U(FN) on Operating Point\n"
        "U(TP)=10, U(TN)=1, U(FP)=-2 held constant",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "eu_sensitivity_analysis.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"Saved → {path}")


def plot_summary_table(scenario_results):
    """Clean comparison table as a figure."""
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")

    headers = ["Deployment", "U(FN)", "τ*", "Recall", "FPR",
               "Attacks\nCaught", "Attacks\nMissed", "False\nAlarms"]
    rows = []
    for r in scenario_results:
        cfg = next(c for c in UTILITY_CONFIGS if c["name"] == r["name"])
        rows.append([
            r["name"],
            str(cfg["U_FN"]),
            f"{r['tau']:.3f}",
            f"{r['recall']:.3f}",
            f"{r['fpr']:.4f}",
            f"{r['tp']:,}",
            f"{r['fn']:,}",
            f"{r['fp']:,}",
        ])

    table = ax.table(
        cellText=rows, colLabels=headers,
        cellLoc="center", loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 2.0)

    # Color header
    for j in range(len(headers)):
        table[0, j].set_facecolor("#4C72B0")
        table[0, j].set_text_props(color="white", fontweight="bold")

    # Color rows by deployment
    for i, r in enumerate(scenario_results):
        for j in range(len(headers)):
            table[i+1, j].set_facecolor(r["color"] + "30")

    ax.set_title(
        "EU Decision Layer — Summary Across Deployment Scenarios\n"
        "Same trained model, different τ*, no retraining",
        fontsize=11, fontweight="bold", pad=20
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "eu_summary_table.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    # Load
    try:
        y_test = np.load(OUTPUT_DIR / "y_test.npy")
        y_prob = np.load(OUTPUT_DIR / "y_prob.npy")
    except FileNotFoundError:
        print("Run train_xgb_final.py first.")
        return

    auroc        = roc_auc_score(y_test, y_prob)
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)

    print(f"Loaded {len(y_test):,} test samples")
    print(f"Model AUROC: {auroc:.4f}")

    # Print τ* for each scenario
    print("\n── EU Threshold Derivation ──────────────────────────────────────")
    print(f"  Formula: τ* = (U(TN)-U(FP)) / (U(TP)-U(FN)+U(TN)-U(FP))")
    print(f"  Fixed:   U(TP)=10  U(TN)=1  U(FP)=-2")
    print(f"\n  {'Deployment':<22}  {'U(FN)':>6}  {'τ*':>7}  {'τ* %':>6}")
    print(f"  {'─'*50}")
    for cfg in UTILITY_CONFIGS:
        tau = compute_tau(cfg["U_TP"], cfg["U_TN"], cfg["U_FP"], cfg["U_FN"])
        print(f"  {cfg['name']:<22}  {cfg['U_FN']:>6}  "
              f"{tau:>7.4f}  {tau*100:>5.2f}%")

    # Evaluate each scenario
    print("\n── Scenario Results ─────────────────────────────────────────────")
    scenario_results = []
    for cfg in UTILITY_CONFIGS:
        result = evaluate_scenario(cfg, y_test, y_prob, fpr, tpr, thresholds)
        scenario_results.append(result)

    # Plots
    print("\n── Generating plots ─────────────────────────────────────────────")
    plot_roc_with_operating_points(fpr, tpr, auroc, scenario_results)
    plot_tau_vs_fn_cost(scenario_results)
    plot_summary_table(scenario_results)

    print(f"\nAll outputs saved to ./{OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

Loaded 102,106 test samples
Model AUROC: 0.9892

── EU Threshold Derivation ──────────────────────────────────────
  Formula: τ* = (U(TN)-U(FP)) / (U(TP)-U(FN)+U(TN)-U(FP))
  Fixed:   U(TP)=10  U(TN)=1  U(FP)=-2

  Deployment               U(FN)       τ*    τ* %
  ──────────────────────────────────────────────────
  Military Drone            -200   0.0141   1.41%
  Civilian UAV               -50   0.0476   4.76%
  Survey Drone               -10   0.1304  13.04%
  Research / Low Risk         -5   0.1667  16.67%

── Scenario Results ─────────────────────────────────────────────

───────────────────────────────────────────────────────
 Military Drone
 Missing attack is catastrophic — very low threshold
───────────────────────────────────────────────────────
  Utility values : U(TP)=   10  U(TN)=   1  U(FP)=  -2  U(FN)= -200
  τ* (threshold) : 0.0141
  E[Alarm]       : 1.0745
  E[Pass]        : -50.4979
  Decision       : alarm if P(attack) ≥ 0.0141

  Results:
    TP (attacks caught)   : 

In [ ]:
"""
GPS Spoofing Detection — Full System Test
Tests the final XGBoost + EU system with three experiments:

Experiment 1 — Standard evaluation
  EU threshold vs fixed threshold on held-out test set.
  Shows τ* correctly tracks optimal operating point per scenario.

Experiment 2 — Holdout attack test
  Train on 2 attack types, test whether model detects the 3rd.
  Simulates encountering an attack type not seen during training.
  Runs 3 times: hold out simplistic, intermediate, sophisticated.

Experiment 3 — EU sensitivity sweep
  Sweeps U(FN) from -2 to -500.
  Shows how τ*, recall, FPR change continuously.
  Key paper figure: demonstrates deployment tunability.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)
import xgboost as xgb

DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("final_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
FEATURES     = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]

ATTACK_NAMES = {1: "Simplistic", 2: "Intermediate", 3: "Sophisticated"}

UTILITY_CONFIGS = [
    {"name": "Military Drone",    "U_TP": 10, "U_TN": 1, "U_FP": -2, "U_FN": -200},
    {"name": "Civilian UAV",      "U_TP": 10, "U_TN": 1, "U_FP": -2, "U_FN": -50},
    {"name": "Survey Drone",      "U_TP": 10, "U_TN": 1, "U_FP": -2, "U_FN": -10},
    {"name": "Research / Low Risk","U_TP": 10, "U_TN": 1, "U_FP": -2, "U_FN": -5},
]


# ── EU core ───────────────────────────────────────────────────────────────────
def compute_tau(U_TP, U_TN, U_FP, U_FN) -> float:
    return (U_TN - U_FP) / (U_TP - U_FN + U_TN - U_FP)


def eu_metrics(y_test, y_prob, tau):
    y_pred        = (y_prob >= tau).astype(int)
    cm            = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    recall        = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision     = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr_val       = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "recall": recall, "precision": precision, "fpr": fpr_val}


# ── Data loading ──────────────────────────────────────────────────────────────
def load_raw():
    p  = Path(DATA_PATH)
    df = pd.read_excel(p, engine="openpyxl") if p.suffix == ".xlsx" else pd.read_csv(p)
    if "Output" in df.columns:
        df = df.rename(columns={"Output": "label"})
    return df


def load_artifacts():
    model = xgb.XGBClassifier()
    model.load_model(OUTPUT_DIR / "xgb_final.ubj")
    y_test = np.load(OUTPUT_DIR / "y_test.npy")
    y_prob = np.load(OUTPUT_DIR / "y_prob.npy")
    print(f"Loaded model + test set ({len(y_test):,} samples)")
    return model, y_test, y_prob


# ── Experiment 1 — Standard EU evaluation ────────────────────────────────────
def experiment1_standard(y_test, y_prob):
    print("\n" + "="*65)
    print(" EXPERIMENT 1 — EU THRESHOLD vs FIXED THRESHOLD (τ=0.5)")
    print("="*65)

    auroc = roc_auc_score(y_test, y_prob)
    print(f"\n  Model AUROC: {auroc:.4f}")

    # Fixed threshold baseline
    fixed = eu_metrics(y_test, y_prob, 0.5)
    print(f"\n  Fixed threshold (τ=0.5):")
    print(f"    Recall={fixed['recall']:.4f}  "
          f"Precision={fixed['precision']:.4f}  "
          f"FPR={fixed['fpr']:.4f}  "
          f"FN={fixed['fn']:,}  FP={fixed['fp']:,}")

    # EU scenarios
    print(f"\n  EU adaptive thresholds:")
    print(f"  {'Deployment':<22}  {'τ*':>6}  {'Recall':>7}  "
          f"{'Precision':>9}  {'FPR':>7}  {'FN':>6}  {'FP':>7}")
    print(f"  {'─'*70}")

    results = []
    for cfg in UTILITY_CONFIGS:
        tau = compute_tau(cfg["U_TP"], cfg["U_TN"], cfg["U_FP"], cfg["U_FN"])
        m   = eu_metrics(y_test, y_prob, tau)
        print(f"  {cfg['name']:<22}  {tau:>6.4f}  {m['recall']:>7.4f}  "
              f"{m['precision']:>9.4f}  {m['fpr']:>7.4f}  "
              f"{m['fn']:>6,}  {m['fp']:>7,}")
        results.append({**cfg, "tau": tau, **m})

    return results, auroc


# ── Experiment 2 — Holdout attack test ───────────────────────────────────────
def experiment2_holdout(df):
    """
    Train on legitimate + 2 attack types.
    Test whether model detects the 3rd (held-out) attack type.
    Evaluates generalization across attack types.
    """
    print("\n" + "="*65)
    print(" EXPERIMENT 2 — HOLDOUT ATTACK TEST")
    print(" (train on 2 attack types, detect the 3rd)")
    print("="*65)

    holdout_results = []

    for held_out in [1, 2, 3]:
        train_attacks = [a for a in [1, 2, 3] if a != held_out]
        print(f"\n  Holding out: {ATTACK_NAMES[held_out]}")
        print(f"  Training on: {[ATTACK_NAMES[a] for a in train_attacks]}")

        # Build training set: legitimate + 2 attack types
        mask_train = df["label"].astype(int).isin([0] + train_attacks)
        df_train   = df[mask_train].copy()
        y_tr       = (df_train["label"].astype(int) != 0).astype(int).values
        X_tr       = df_train[FEATURES].values.astype(np.float32)

        # Build test set: legitimate + held-out attack only
        mask_test = df["label"].astype(int).isin([0, held_out])
        df_test   = df[mask_test].copy()
        y_te      = (df_test["label"].astype(int) != 0).astype(int).values
        X_te      = df_test[FEATURES].values.astype(np.float32)

        # Train/val split on training data
        X_tv, X_val_, y_tv, y_val_ = train_test_split(
            X_tr, y_tr, test_size=0.20, stratify=y_tr, random_state=RANDOM_STATE
        )

        scale_pw = (y_tv == 0).sum() / max((y_tv == 1).sum(), 1)

        model_h = xgb.XGBClassifier(
            n_estimators          = 500,
            max_depth             = 6,
            learning_rate         = 0.05,
            subsample             = 0.8,
            colsample_bytree      = 0.8,
            objective             = "binary:logistic",
            eval_metric           = "logloss",
            scale_pos_weight      = scale_pw,
            early_stopping_rounds = 20,
            tree_method           = "hist",
            random_state          = RANDOM_STATE,
            n_jobs                = -1,
            verbosity             = 0,
        )
        model_h.fit(
            X_tv, y_tv,
            eval_set=[(X_tv, y_tv), (X_val_, y_val_)],
            verbose=False,
        )

        y_prob_h = model_h.predict_proba(X_te)[:, 1]
        auroc_h  = roc_auc_score(y_te, y_prob_h)

        # EU at civilian UAV setting
        tau = compute_tau(10, 1, -2, -50)
        m   = eu_metrics(y_te, y_prob_h, tau)

        print(f"  AUROC on {ATTACK_NAMES[held_out]:<15}: {auroc_h:.4f}  "
              f"|  Recall={m['recall']:.4f}  FPR={m['fpr']:.4f}  "
              f"FN={m['fn']:,}")

        holdout_results.append({
            "held_out": ATTACK_NAMES[held_out],
            "auroc":    auroc_h,
            **m,
        })

    return holdout_results


# ── Experiment 3 — EU sensitivity sweep ──────────────────────────────────────
def experiment3_sensitivity(y_test, y_prob):
    """
    Sweeps U(FN) continuously and tracks τ*, recall, FPR.
    Shows the deployment tunability claim quantitatively.
    """
    print("\n" + "="*65)
    print(" EXPERIMENT 3 — EU SENSITIVITY SWEEP")
    print(" (sweeping U(FN) from -2 to -500)")
    print("="*65)

    fn_costs = np.concatenate([
        np.linspace(2, 10, 20),
        np.linspace(10, 100, 30),
        np.linspace(100, 500, 20),
    ])

    taus     = []
    recalls  = []
    fprs     = []
    precisions = []

    for c in fn_costs:
        tau = compute_tau(10, 1, -2, -c)
        m   = eu_metrics(y_test, y_prob, tau)
        taus.append(tau)
        recalls.append(m["recall"])
        fprs.append(m["fpr"])
        precisions.append(m["precision"])

    # Print key points
    print(f"\n  {'|U(FN)|':>8}  {'τ*':>7}  {'Recall':>7}  {'FPR':>7}  {'Precision':>10}")
    print(f"  {'─'*48}")
    for c in [5, 10, 20, 50, 100, 200, 500]:
        tau = compute_tau(10, 1, -2, -c)
        m   = eu_metrics(y_test, y_prob, tau)
        print(f"  {c:>8}  {tau:>7.4f}  {m['recall']:>7.4f}  "
              f"{m['fpr']:>7.4f}  {m['precision']:>10.4f}")

    return fn_costs, taus, recalls, fprs, precisions


# ── Plots ─────────────────────────────────────────────────────────────────────
def plot_experiment1(y_test, y_prob, eu_results, auroc):
    fpr_c, tpr_c, thresholds = roc_curve(y_test, y_prob)

    colors = ["#D32F2F", "#F57C00", "#388E3C", "#1565C0"]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.plot(fpr_c, tpr_c, color="#4C72B0", lw=2.5,
            label=f"XGBoost  AUROC={auroc:.4f}", zorder=2)
    ax.plot([0,1], [0,1], "k--", lw=1, zorder=1)

    # Fixed threshold point
    m_fixed = eu_metrics(y_test, y_prob, 0.5)
    ax.scatter(m_fixed["fpr"], m_fixed["recall"], marker="x",
               color="black", s=150, zorder=6, linewidths=2,
               label=f"Fixed τ=0.5  R={m_fixed['recall']:.3f}  FPR={m_fixed['fpr']:.3f}")

    for r, color in zip(eu_results, colors):
        ax.scatter(r["fpr"], r["recall"], color=color,
                   s=120, zorder=5,
                   label=f"{r['name']}  τ*={r['tau']:.3f}  "
                         f"R={r['recall']:.3f}  FPR={r['fpr']:.3f}")
        ax.annotate(r["name"],
                    xy=(r["fpr"], r["recall"]),
                    xytext=(r["fpr"]+0.01, r["recall"]-0.025),
                    fontsize=8, color=color)

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate (Recall)")
    ax.set_title("Experiment 1 — EU Operating Points on ROC Curve\n"
                 "Each τ* derived analytically from deployment costs")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlim(-0.005, 0.30)
    ax.set_ylim(0.70, 1.005)
    plt.tight_layout()
    path = OUTPUT_DIR / "test_exp1_roc.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"  Saved → {path}")


def plot_experiment2(holdout_results):
    names  = [r["held_out"] for r in holdout_results]
    aurocs = [r["auroc"]    for r in holdout_results]
    recalls = [r["recall"]  for r in holdout_results]
    fprs    = [r["fpr"]     for r in holdout_results]

    x      = np.arange(len(names))
    colors = ["#FF5722", "#FF9800", "#9C27B0"]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    for ax, vals, title, ylabel in zip(
        axes,
        [aurocs, recalls, fprs],
        ["AUROC on held-out attack",
         "Recall on held-out attack",
         "FPR on held-out attack"],
        ["AUROC", "Recall", "FPR"],
    ):
        bars = ax.bar(names, vals, color=colors, alpha=0.85)
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1.1)
        ax.axhline(0.5, color="gray", ls="--", lw=1, label="Random")

    fig.suptitle(
        "Experiment 2 — Holdout Attack Test\n"
        "Model trained on legitimate + 2 attack types, tested on 3rd",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "test_exp2_holdout.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"  Saved → {path}")


def plot_experiment3(fn_costs, taus, recalls, fprs, precisions):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, vals, ylabel, color, title in zip(
        axes,
        [taus, recalls, fprs],
        ["τ* (alarm threshold)", "Recall", "FPR"],
        ["#4C72B0", "#DD8452", "#2CA02C"],
        ["τ* vs |U(FN)|\n(higher cost → lower threshold)",
         "Recall vs |U(FN)|\n(higher cost → catch more attacks)",
         "FPR vs |U(FN)|\n(higher cost → more false alarms)"],
    ):
        ax.plot(fn_costs, vals, color=color, lw=2)
        ax.set_xlabel("|U(FN)|  (false negative cost)")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xscale("log")

        # Mark 4 deployment scenarios
        for cfg, marker in zip(UTILITY_CONFIGS, ["v","o","s","^"]):
            c_val = abs(cfg["U_FN"])
            tau   = compute_tau(cfg["U_TP"], cfg["U_TN"], cfg["U_FP"], cfg["U_FN"])
            m     = eu_metrics(
                np.load(OUTPUT_DIR / "y_test.npy"),
                np.load(OUTPUT_DIR / "y_prob.npy"),
                tau,
            )
            plot_val = {"τ* (alarm threshold)": tau,
                        "Recall": m["recall"],
                        "FPR": m["fpr"]}[ylabel]
            ax.scatter(c_val, plot_val, marker=marker, s=80,
                       color="red", zorder=5,
                       label=cfg["name"].split()[0])

        if ax == axes[0]:
            ax.legend(fontsize=7)

    fig.suptitle(
        "Experiment 3 — EU Sensitivity Sweep\n"
        "U(TP)=10, U(TN)=1, U(FP)=-2  |  Red markers = 4 deployment scenarios",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "test_exp3_sensitivity.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"  Saved → {path}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("Loading artifacts …")
    model, y_test, y_prob = load_artifacts()
    df = load_raw()

    # Experiment 1
    eu_results, auroc = experiment1_standard(y_test, y_prob)

    # Experiment 2
    holdout_results = experiment2_holdout(df)

    # Experiment 3
    fn_costs, taus, recalls, fprs, precisions = experiment3_sensitivity(
        y_test, y_prob
    )

    # Plots
    print("\n── Generating plots ─────────────────────────────────────────────")
    plot_experiment1(y_test, y_prob, eu_results, auroc)
    plot_experiment2(holdout_results)
    plot_experiment3(fn_costs, taus, recalls, fprs, precisions)

    # Final summary
    print(f"""
══════════════════════════════════════════════════════════════════
 SYSTEM TEST SUMMARY
══════════════════════════════════════════════════════════════════
 Model        : Binary XGBoost, 9 physics features
 AUROC        : {auroc:.4f}

 Experiment 1 — EU vs Fixed Threshold
   Fixed τ=0.5 : Recall={eu_metrics(y_test,y_prob,0.5)['recall']:.4f}  FPR={eu_metrics(y_test,y_prob,0.5)['fpr']:.4f}
   EU military : τ*={compute_tau(10,1,-2,-200):.4f}  → higher recall, higher FPR
   EU research : τ*={compute_tau(10,1,-2,-5):.4f}  → lower recall, lower FPR

 Experiment 2 — Holdout Attack Test
   {holdout_results[0]['held_out']:<15}: AUROC={holdout_results[0]['auroc']:.4f}
   {holdout_results[1]['held_out']:<15}: AUROC={holdout_results[1]['auroc']:.4f}
   {holdout_results[2]['held_out']:<15}: AUROC={holdout_results[2]['auroc']:.4f}

 Experiment 3 — EU Sensitivity Sweep
   τ* ranges from {min(taus):.4f} to {max(taus):.4f} across U(FN) sweep
   Recall ranges from {min(recalls):.4f} to {max(recalls):.4f}
   FPR    ranges from {min(fprs):.4f} to {max(fprs):.4f}

 Outputs saved to ./final_outputs/
   test_exp1_roc.png         — ROC with EU operating points
   test_exp2_holdout.png     — Holdout attack generalization
   test_exp3_sensitivity.png — EU sensitivity sweep
══════════════════════════════════════════════════════════════════
""")


if __name__ == "__main__":
    main()

Loading artifacts …
Loaded model + test set (102,106 samples)

 EXPERIMENT 1 — EU THRESHOLD vs FIXED THRESHOLD (τ=0.5)

  Model AUROC: 0.9892

  Fixed threshold (τ=0.5):
    Recall=0.9912  Precision=0.8039  FPR=0.0685  FN=198  FP=5,449

  EU adaptive thresholds:
  Deployment                  τ*   Recall  Precision      FPR      FN       FP
  ──────────────────────────────────────────────────────────────────────
  Military Drone          0.0141   1.0000     0.5352   0.2461       0   19,578
  Civilian UAV            0.0476   1.0000     0.6701   0.1395       0   11,099
  Survey Drone            0.1304   0.9997     0.7524   0.0932       7    7,415
  Research / Low Risk     0.1667   0.9996     0.7638   0.0876       9    6,966

 EXPERIMENT 2 — HOLDOUT ATTACK TEST
 (train on 2 attack types, detect the 3rd)

  Holding out: Simplistic
  Training on: ['Intermediate', 'Sophisticated']
  AUROC on Simplistic     : 0.6148  |  Recall=0.4371  FPR=0.1960  FN=20,521

  Holding out: Intermediate
  Traini

In [ ]:
# ── PCA Reconstruction Error Detector (Linear Autoencoder) ──
# "The Zero-Day Net" - Catches unnatural physics correlations with near-zero CPU cost.

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
from pathlib import Path

# ── 1. Configuration ──
FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]
DATA_PATH = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("pca_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

N_COMPONENTS = 3  # Compress 9 features down to 3 to force a bottleneck
THRESHOLD_PERCENTILE = 95 # Sets trigger threshold at the 95th percentile of normal noise

def main():
    # ── 2. Load and Prep Data ──
    print(f"Loading {len(FEATURES)} features...")
    df = pd.read_excel(DATA_PATH)

    # Binary Labels: 0 = Legitimate, 1 = Attack
    df['label'] = df['Output'].apply(lambda x: 0 if x == 0 else 1)

    X = df[FEATURES].values.astype(np.float32)
    y = df['label'].values

    # ── 3. Split Data ──
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Filter: Train PCA ONLY on Legitimate data (Class 0)
    X_train_legit = X_train[y_train == 0]

    # ── 4. Scale Features (Mandatory for PCA) ──
    # PCA relies on variance, so all features must share the same scale
    scaler = StandardScaler()
    X_train_legit_scaled = scaler.fit_transform(X_train_legit)
    X_test_scaled = scaler.transform(X_test)

    # ── 5. Train PCA (Learn the Physics Correlations) ──
    print(f"Training PCA on {len(X_train_legit_scaled):,} legitimate samples...")
    pca = PCA(n_components=N_COMPONENTS, random_state=42)
    pca.fit(X_train_legit_scaled)

    explained_variance = sum(pca.explained_variance_ratio_)
    print(f"Variance Retained: {explained_variance*100:.1f}% using just {N_COMPONENTS} components.")

    # ── 6. Determine Safe Threshold ──
    # We reconstruct the healthy training data to see what "normal" error looks like
    X_train_compressed = pca.transform(X_train_legit_scaled)
    X_train_reconstructed = pca.inverse_transform(X_train_compressed)

    # Calculate Mean Squared Error (MSE) per sample
    train_reconstruction_error = np.mean(np.square(X_train_legit_scaled - X_train_reconstructed), axis=1)

    # Set the alarm threshold dynamically based on normal flight noise
    error_threshold = np.percentile(train_reconstruction_error, THRESHOLD_PERCENTILE)
    print(f"Calculated Error Threshold (95th Percentile): {error_threshold:.4f}")

    # ── 7. Inference on Test Set (Simulation of Drone Flight) ──
    print("\nRunning inference on mixed test set (Legitimate + Attacks)...")
    X_test_compressed = pca.transform(X_test_scaled)
    X_test_reconstructed = pca.inverse_transform(X_test_compressed)

    test_reconstruction_error = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

    # If the error is higher than our healthy threshold, flag as Attack (1)
    y_pred = (test_reconstruction_error > error_threshold).astype(int)

    # ── 8. Evaluation ──
    auroc = roc_auc_score(y_test, test_reconstruction_error)
    print("\n" + "="*40)
    print(f"--- PCA Reconstruction Performance (AUROC: {auroc:.4f}) ---")
    print("="*40)
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Attack"]))

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(f"  True Negatives (Safe flights continued)   : {cm[0][0]}")
    print(f"  False Positives (Nuisance Aborts)         : {cm[0][1]}  <-- Managed by {THRESHOLD_PERCENTILE}th percentile")
    print(f"  False Negatives (Attacks that got through): {cm[1][0]}")
    print(f"  True Positives (Attacks defeated)         : {cm[1][1]}")

    # ── 9. Show the CPU Math for the Paper ──
    print("\n" + "-"*50)
    print("DRONE CPU DEPLOYMENT COST:")
    print("Unlike a Neural Network, this runs in pure linear algebra:")
    print(f" 1. Compress  : X_scaled * ({len(FEATURES)}x{N_COMPONENTS} matrix)")
    print(f" 2. Rebuild   : Compressed * ({N_COMPONENTS}x{len(FEATURES)} matrix)")
    print(f" Total Compute: Just {(len(FEATURES) * N_COMPONENTS) * 2} multiplication operations per inference.")
    print("-"*50)

    # ── 10. Save Artifacts ──
    joblib.dump(pca, OUTPUT_DIR / "pca_model.pkl")
    joblib.dump(scaler, OUTPUT_DIR / "pca_scaler.pkl")
    np.save(OUTPUT_DIR / "pca_threshold.npy", np.array([error_threshold]))
    print(f"\nSaved PCA artifacts to ./{OUTPUT_DIR}/")

if __name__ == "__main__":
    main()

Loading 9 features...
Training PCA on 318,260 legitimate samples...
Variance Retained: 81.5% using just 3 components.
Calculated Error Threshold (95th Percentile): 0.4338

Running inference on mixed test set (Legitimate + Attacks)...

--- PCA Reconstruction Performance (AUROC: 0.5211) ---
              precision    recall  f1-score   support

  Legitimate       0.79      0.95      0.86     79565
      Attack       0.35      0.09      0.15     22541

    accuracy                           0.76    102106
   macro avg       0.57      0.52      0.51    102106
weighted avg       0.69      0.76      0.70    102106

Confusion Matrix:
  True Negatives (Safe flights continued)   : 75639
  False Positives (Nuisance Aborts)         : 3926  <-- Managed by 95th percentile
  False Negatives (Attacks that got through): 20402
  True Positives (Attacks defeated)         : 2139

--------------------------------------------------
DRONE CPU DEPLOYMENT COST:
Unlike a Neural Network, this runs in pure linea